# Step 10 playground — Riemannian frame aggregation (E2 / RQ3)

Three ways to average concept frames, now selectable via `Concept.average(..., method=)`:

- **`extrinsic`** (default, the E0 baseline): polar factor of the weighted sum — the chordal mean.
- **`aligned`**: generalized-Procrustes mean — each frame is first rotated *within its own span* to best match the mean, then chordally averaged. Robust to rotation mismatch between frames.
- **`frechet`**: geodesic Karcher mean on the Stiefel manifold (canonical metric) — the true intrinsic mean, via `stiefel_exp` / `stiefel_log`.

§1–2 are synthetic (CPU, instant); §3 uses real Llama concepts.

In [ ]:
import torch

from frames.linalg.orthogonalization import solve_procrustes
from frames.linalg.stiefel import (
    aligned_mean,
    canonical_norm,
    frechet_mean,
    stiefel_exp,
    stiefel_log,
)
from frames.representations.concept import Concept

D, K = 16, 3


def random_point(seed: int) -> torch.Tensor:
    g = torch.Generator().manual_seed(seed)
    q, _ = torch.linalg.qr(torch.randn(D, K, generator=g, dtype=torch.float64))
    return q

## 1. Geodesics: walking on the Stiefel manifold

`stiefel_exp(x, t * log_x(y))` traces the geodesic from `x` to `y`. Distances along it add up linearly, and the endpoint is recovered exactly.

In [ ]:
x, y = random_point(0), random_point(1)
delta = stiefel_log(x, y)
distance = float(canonical_norm(x, delta))
print(f"geodesic distance d(x, y) = {distance:.4f}")

for t in (0.0, 0.25, 0.5, 0.75, 1.0):
    point = stiefel_exp(x, t * delta)
    d_from_x = float(canonical_norm(x, stiefel_log(x, point))) if t > 0 else 0.0
    ortho_err = float((point.mT @ point - torch.eye(K, dtype=torch.float64)).norm())
    print(
        f"t={t:.2f}: d(x, gamma(t)) = {d_from_x:.4f}  (= t*d: {t * distance:.4f}),  orthonormality error {ortho_err:.1e}"
    )

print(f"\nendpoint recovered: {torch.allclose(stiefel_exp(x, delta), y, atol=1e-8)}")

## 2. Three means, and why alignment matters

The extrinsic mean is *rotation-sensitive*: rotate one input within its own span (which leaves its subspace — and its meaning as a span — unchanged) and the mean moves. The aligned mean re-gauges inputs first, so it lands on the same orbit either way. The Fréchet mean minimizes summed squared *geodesic* distance — for two points it is the geodesic midpoint.

In [ ]:
a, b = random_point(2), random_point(3)
points = torch.stack([a, b])

means = {
    "extrinsic": solve_procrustes(points.mean(0).float()).double(),
    "aligned": aligned_mean(points),
    "frechet": frechet_mean(points),
}

print("geodesic distance to each constituent (should be equal for frechet):")
for name, mean in means.items():
    d_a = float(canonical_norm(mean, stiefel_log(mean, a)))
    d_b = float(canonical_norm(mean, stiefel_log(mean, b)))
    print(f"  {name:>9}: d(mean, a) = {d_a:.4f}   d(mean, b) = {d_b:.4f}")

midpoint = stiefel_exp(a, stiefel_log(a, b) / 2)
print(
    f"\nfrechet mean == geodesic midpoint: {torch.allclose(means['frechet'], midpoint, atol=1e-6)}"
)

In [ ]:
# rotate `a` within its span: same subspace, different Stiefel representative
g = torch.Generator().manual_seed(4)
rotation, _ = torch.linalg.qr(torch.randn(K, K, generator=g, dtype=torch.float64))
rotated = torch.stack([a @ rotation, b])

from frames.linalg.stiefel import _polar

for name, fn in (
    ("extrinsic", lambda p: solve_procrustes(p.mean(0).float()).double()),
    ("aligned", aligned_mean),
):
    plain, moved = fn(points), fn(rotated)
    raw_gap = float((plain - moved).norm())
    regauged = moved @ _polar(moved.mT @ plain)
    orbit_gap = float((plain - regauged).norm())
    print(
        f"{name:>9}: mean moved by {raw_gap:.4f} after rotating one input  (orbit distance: {orbit_gap:.1e})"
    )

The extrinsic mean lands on a genuinely different orbit (orbit distance stays large); the aligned mean returns the same frame up to gauge. FRH's trace scoring is gauge-*sensitive*, so the returned representative matters — `aligned_mean` anchors it at the extrinsic-mean initialization, the gauge that won E0.

## 3. Real concepts

Same comparison on Llama-3.1-8B concept frames, including the Step-8 composition probe: does `mean(woman, child)` approach `girl`?

In [ ]:
from frames.representations import FrameUnembeddingRepresentation

fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)
MIN_LEMMAS, MAX_TOKENS = 11, 3

In [ ]:
joy = fur.get_concept_cached("joy.n.01", MIN_LEMMAS, MAX_TOKENS)
dog = fur.get_concept_cached("dog.n.01", MIN_LEMMAS, MAX_TOKENS)

print(f"rho(joy, dog) = {float(joy.rho(dog)):.4f}\n")
for method in ("extrinsic", "aligned", "frechet"):
    mean = Concept.average([joy, dog], method=method)
    print(
        f"{method:>9}: rho(mean, joy) = {float(mean.rho(joy)):.4f}   rho(mean, dog) = {float(mean.rho(dog)):.4f}"
    )

In [ ]:
woman = fur.get_concept_cached("woman.n.01", MIN_LEMMAS, MAX_TOKENS)
child = fur.get_concept_cached("child.n.01", MIN_LEMMAS, MAX_TOKENS)
girl = fur.get_concept_cached("girl.n.01", MIN_LEMMAS, MAX_TOKENS)

print("composition probe: rho(mean(woman, child), girl)")
print(
    f"  constituents: rho(woman, girl) = {float(woman.rho(girl)):.4f}, rho(child, girl) = {float(child.rho(girl)):.4f}\n"
)
for method in ("extrinsic", "aligned", "frechet"):
    mean = Concept.average([woman, child], method=method)
    print(f"{method:>9}: rho(mean, girl) = {float(mean.rho(girl)):.4f}")

## 4. Try it in generation

`quick_generate_with_topk_mean_frame_guide` now takes `average_method=`; anything the harness accepts works with all three means. (Kept as a snippet — run `18_e2_riemannian.ipynb` for the full comparison.)

```python
texts, probe = fur.quick_generate_with_topk_mean_frame_guide(
    ["<|start_header_id|>user<|end_header_id|>Tell me a short story.<|eot_id|>"
     "<|start_header_id|>assistant<|end_header_id|>Once"],
    guides=[("joy.n.01",), ("dog.n.01",)],
    min_lemmas_per_synset=11,
    max_token_count=3,
    k=4,
    steps=24,
    average_method="frechet",
)
```